# Cortex M1: free-GPU QLoRA workflow
This notebook has been prepared and syntax-checked, not GPU-executed. In your own Google Colab account choose a free GPU runtime. Availability, memory and quotas are not guaranteed; do not select paid compute. Training downloads several GB of public model/dependency files. Stop if no GPU is allocated.
Only M1 is trained. The starter dataset is small synthetic smoke-test material, not evidence of reliable routing. Review and expand it before claiming quality. Keep the test split untouched while choosing prompts or training settings.
Upload these six local files when prompted: `m1.py`, `dataset.py`, `train_m1.py`, `evaluate_m1.py`, `requirements-training.txt`, and `data/starter.json` (uploaded as `starter.json`). Do not upload credentials or private documents. The notebook never uploads models to a public hub.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import venv
from datetime import datetime, timezone
from google.colab import files
subprocess.run(['nvidia-smi'], check=True)
uploaded = files.upload()
required = ['m1.py', 'dataset.py', 'train_m1.py', 'evaluate_m1.py', 'requirements-training.txt', 'starter.json']
missing = [name for name in required if not Path(name).is_file()]
assert not missing, f'Missing files: {missing}'
DATA = str(Path('starter.json').resolve())


## Isolated training environment
This creates a notebook-local virtual environment, leaving Colab's global Python packages alone. The direct dependencies are pinned; transitive packages are recorded after installation. If GPU preflight fails, stop and resolve that environment rather than starting a CPU training run.


In [ ]:
ENV = Path('cortex-training-env')
if not ENV.exists():
    venv.EnvBuilder(with_pip=True).create(ENV)
PY = str((ENV / 'bin' / 'python').resolve())
subprocess.run([PY, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)
subprocess.run([PY, 'train_m1.py', '--check'], check=True)
subprocess.run([PY, 'dataset.py', DATA], check=True)
REVISION = subprocess.check_output([PY, '-c', "from huggingface_hub import HfApi; print(HfApi().model_info('Qwen/Qwen3-4B').sha)"], text=True).strip()
assert len(REVISION) == 40
print('Pinned Qwen3-4B revision:', REVISION)
RUN = Path('m1-run-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
assert not RUN.exists()
RUN.mkdir()
(RUN / 'environment.freeze.txt').write_text(subprocess.check_output([PY, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')


In [ ]:
records = json.loads(Path(DATA).read_text(encoding='utf-8'))
for row in records:
    if row['split'] == 'train':
        print(json.dumps({'id': row['id'], 'query': row['request']['query'], 'decision': row['decision']}, ensure_ascii=False))
REVIEWED_DATA = False
assert REVIEWED_DATA, 'Review and correct the training labels, then explicitly set REVIEWED_DATA = True.'


## Training smoke test
A two-step run checks model loading, memory, labels and adapter saving. It does not establish model quality. If out of memory, stop, inspect the allocated GPU, and reduce sequence length only after checking that complete examples still fit. The script rejects truncation.


In [ ]:
assert REVIEWED_DATA
subprocess.run([PY, 'train_m1.py', '--data', DATA, '--revision', REVISION, '--reviewed-data', '--max-steps', '2', '--output', str(RUN / 'smoke')], check=True)


## Fair baseline
Use the same Hugging Face base revision, NF4 loading, prompt and decoding as adapter evaluation. Ollama's quantization and constrained JSON decoding differ, so an Ollama comparison alone would not isolate the effect of fine-tuning.


In [ ]:
subprocess.run([PY, 'evaluate_m1.py', '--data', DATA, '--backend', 'hf-base', '--revision', REVISION, '--split', 'eval', '--output', str(RUN / 'baseline-eval.json')], check=True)


In [ ]:
assert REVIEWED_DATA
subprocess.run([PY, 'train_m1.py', '--data', DATA, '--revision', REVISION, '--reviewed-data', '--epochs', '1', '--output', str(RUN / 'trained')], check=True)
ADAPTER = str(RUN / 'trained' / 'adapter')
subprocess.run([PY, 'evaluate_m1.py', '--data', DATA, '--backend', 'hf-adapter', '--adapter', ADAPTER, '--split', 'eval', '--output', str(RUN / 'adapter-eval.json')], check=True)
subprocess.run([PY, 'evaluate_m1.py', '--compare', str(RUN / 'baseline-eval.json'), str(RUN / 'adapter-eval.json')], check=True)


## Held-out test and export
Run this only after freezing the candidate. Do not repeatedly tune against these test labels. Inspect generated specialist instructions as well as numerical routing scores. A trained adapter is not automatically an improvement; retain the baseline if it is better. Real teammate integration is separate.


In [ ]:
for backend, extra, filename in [('hf-base', ['--revision', REVISION], 'baseline-test.json'), ('hf-adapter', ['--adapter', ADAPTER], 'adapter-test.json')]:
    subprocess.run([PY, 'evaluate_m1.py', '--data', DATA, '--backend', backend, *extra, '--split', 'test', '--output', str(RUN / filename)], check=True)
subprocess.run([PY, 'evaluate_m1.py', '--compare', str(RUN / 'baseline-test.json'), str(RUN / 'adapter-test.json')], check=True)
import shutil
snapshot = RUN / 'source'
snapshot.mkdir(exist_ok=False)
for name in required:
    shutil.copy2(name, snapshot / name)
archive = shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
files.download(archive)
